# NASA Kepler Objects of Interest - Classification

This notebook prepares the Review 1 classification track for the 23CSE301 Machine Learning Capstone Project.

Implemented in this step:

- Dataset audit and Review 1 EDA
- Leakage and identifier review
- Train/test split with `random_state=42`
- Leakage-safe preprocessing pipelines
- Gaussian Naive Bayes
- Decision Tree Classifier, including tuning
- Shared comparison table for the implemented models

Academic note: the notebook intentionally leaves interpretation TODOs for the student team. Do not submit generated scientific conclusions without checking the outputs yourself.

## 1. Problem Statement

The goal is to classify Kepler Objects of Interest (KOIs) into their official disposition classes using tabular measurements from the NASA Kepler KOI catalog.

The model should predict the selected disposition/status field from non-leaking measurement features. The natural problem is multiclass classification because the dataset contains more than two disposition labels.

## 2. Dataset Description

Dataset: NASA Kepler Objects of Interest (KOI)

Official NASA Open Data Portal page: https://data.nasa.gov/dataset/kepler-objects-of-interest-koi

Programmatic CSV source used here: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv

The NASA page describes KOI as a catalog of observed Kepler targets that are flagged as possible exoplanet detections, while also noting that some entries may be false positives.

## 3. Imports and Reproducibility

In [ ]:
from pathlib import Path
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

## 4. Data Loading

In [ ]:
DATA_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv"


def find_project_root() -> Path:
    """Return the repository root whether the notebook is run from root or notebooks/."""
    current = Path.cwd().resolve()

    if (current / "data").exists() and (current / "notebooks").exists():
        return current

    if current.name == "notebooks" and (current.parent / "data").exists():
        return current.parent

    for parent in current.parents:
        if (parent / "data").exists() and (parent / "notebooks").exists():
            return parent

    raise FileNotFoundError("Could not locate the project root containing data/ and notebooks/.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "kepler_koi.csv"


def load_koi_data(path: Path = DATA_PATH, url: str = DATA_URL) -> pd.DataFrame:
    """Load the KOI dataset from data/ first, then download it if needed."""
    if path.exists():
        print(f"Loading local dataset: {path}")
        return pd.read_csv(path)

    print("Local dataset not found. Downloading from the Exoplanet Archive...")
    data = pd.read_csv(url)
    path.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(path, index=False)
    print(f"Saved local copy to: {path}")
    return data


df = load_koi_data()

## 5. Dataset Audit

In [ ]:
# First look at the data exactly as loaded.
display(df.head())
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

In [ ]:
print("Column names:")
for index, column in enumerate(df.columns, start=1):
    print(f"{index:03d}. {column}")

In [ ]:
info_buffer = io.StringIO()
df.info(buf=info_buffer)
print(info_buffer.getvalue())

In [ ]:
def build_audit_table(data: pd.DataFrame) -> pd.DataFrame:
    """Create one compact audit table for types, missingness, and cardinality."""
    audit = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "missing_count": data.isna().sum(),
        "missing_percent": data.isna().mean().mul(100).round(2),
        "unique_values": data.nunique(dropna=True),
    })
    return audit.sort_values(["missing_percent", "unique_values"], ascending=[False, False])


audit_table = build_audit_table(df)
display(audit_table)

In [ ]:
print(f"Number of exact duplicate rows: {df.duplicated().sum():,}")

display(df.describe(include="number").T)
display(df.describe(include="object").T)

In [ ]:
target_candidate_columns = [
    column
    for column in df.columns
    if any(token in column.lower() for token in ["disposition", "status", "class", "label"])
]

candidate_summary = []
for column in target_candidate_columns:
    values = df[column].dropna().astype(str).unique().tolist()
    candidate_summary.append({
        "column": column,
        "dtype": str(df[column].dtype),
        "missing_count": int(df[column].isna().sum()),
        "unique_count": int(df[column].nunique(dropna=True)),
        "sample_values": values[:8],
    })

display(pd.DataFrame(candidate_summary))

## 6. Exploratory Data Analysis

This section focuses on readable Review 1 EDA. When there are many numeric columns, the notebook uses a practical subset for plots and keeps summary tables for the full dataset.

In [ ]:
important_numeric_candidates = [
    "koi_period",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "koi_smass",
]

available_important_numeric = [
    column for column in important_numeric_candidates if column in df.columns
]

print("Important numeric columns available for detailed EDA:")
print(available_important_numeric)

display(df[available_important_numeric].describe().T)

In [ ]:
if available_important_numeric:
    plot_columns = available_important_numeric[:9]
    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    axes = axes.ravel()

    for axis, column in zip(axes, plot_columns):
        sns.histplot(df[column], kde=True, ax=axis, color="#2563eb")
        axis.set_title(column)

    for axis in axes[len(plot_columns):]:
        axis.set_visible(False)

    fig.suptitle("Numerical Feature Distributions", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
if available_important_numeric:
    plot_columns = available_important_numeric[:9]
    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    axes = axes.ravel()

    for axis, column in zip(axes, plot_columns):
        sns.boxplot(x=df[column], ax=axis, color="#10b981")
        axis.set_title(column)

    for axis in axes[len(plot_columns):]:
        axis.set_visible(False)

    fig.suptitle("Boxplots for Potential Outlier Review", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def build_iqr_summary(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        series = data[column].dropna()
        if series.empty:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()

        rows.append({
            "feature": column,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "potential_outliers": int(outlier_count),
            "outlier_percent": round(outlier_count / len(series) * 100, 2),
        })
    return pd.DataFrame(rows)


iqr_summary = build_iqr_summary(df, available_important_numeric)
display(iqr_summary)

Extreme astronomical measurements are not automatically errors. Review possible outliers first; remove values only when there is evidence that the value is invalid or was loaded incorrectly.

## 7. Target Definition

In [ ]:
if "koi_disposition" not in df.columns:
    raise KeyError("Expected target column 'koi_disposition' was not found. Re-check the dataset audit.")

target_column = "koi_disposition"
target_classes = sorted(df[target_column].dropna().astype(str).unique().tolist())

print(f"Selected classification target: {target_column}")
print(f"Target classes: {target_classes}")

class_counts = df[target_column].value_counts(dropna=False)
class_percentages = df[target_column].value_counts(normalize=True, dropna=False).mul(100).round(2)

target_distribution = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages,
})

display(target_distribution)

`koi_disposition` is suitable as the classification target because it is the dataset field that stores the KOI disposition/status labels. The values are kept as their real labels, so this remains a natural multiclass classification task instead of being forced into a binary problem.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x=target_column, order=class_counts.index, palette="Set2")
plt.title("Target Distribution: KOI Disposition")
plt.xlabel("KOI disposition")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display(target_distribution)

> STUDENT INTERPRETATION TODO: Explain whether the target classes are balanced enough for model training, and why weighted metrics are useful here.

## 8. Data Cleaning

In [ ]:
working_df = df.copy()

duplicate_count = working_df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count:,}")

if duplicate_count > 0:
    duplicate_rows = working_df[working_df.duplicated(keep=False)]
    display(duplicate_rows.head())
    working_df = working_df.drop_duplicates().copy()
    print(f"Removed {duplicate_count:,} exact duplicate rows.")
else:
    print("No exact duplicate rows were removed.")

In [ ]:
missing_summary = (
    working_df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda table: table["missing_count"].div(len(working_df)).mul(100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

display(missing_summary)

Missing values are not removed with a blanket `dropna()`. The modelling pipeline imputes missing values after the train/test split:

- Numeric features: median imputation
- Categorical features: most-frequent imputation

These are sensible Review 1 defaults because they are simple, explainable, and learned from the training data only.

## 9. Feature Selection

### Feature Engineering

In [ ]:
engineered_features = []

if {"koi_depth", "koi_duration"}.issubset(working_df.columns):
    duration = working_df["koi_duration"].replace(0, np.nan)
    working_df["transit_depth_per_hour"] = working_df["koi_depth"] / duration
    engineered_features.append("transit_depth_per_hour")

print("Engineered features:", engineered_features if engineered_features else "None created")

if engineered_features:
    display(working_df[engineered_features].describe().T)

`transit_depth_per_hour = koi_depth / koi_duration` summarizes how concentrated the observed transit-depth signal is over the measured transit duration.

> STUDENT JUSTIFICATION TODO: Explain why this engineered feature may help distinguish KOI classes after reviewing the actual model outputs.

### Leakage and Identifier Review

In [ ]:
manual_exclusion_reasons = {
    target_column: "selected target column",
    "kepid": "Kepler target identifier, not a measurement feature",
    "kepoi_name": "KOI identifier, not a measurement feature",
    "kepler_name": "confirmed Kepler planet name; can reveal confirmed status",
    "ra_str": "string duplicate of right ascension; numeric ra is available",
    "dec_str": "string duplicate of declination; numeric dec is available",
    "koi_delivname": "catalog delivery metadata",
    "koi_vet_stat": "vetting workflow metadata",
    "koi_quarters": "observing-quarter bitmask metadata; unsuitable for this baseline",
    "koi_pdisposition": "alternate/preliminary disposition closely related to the target",
    "koi_limbdark_mod": "model/provenance text, not a direct measured feature",
    "koi_trans_mod": "transit-model provenance text, not a direct measured feature",
    "koi_sparprov": "stellar-parameter provenance metadata",
    "koi_comment": "vetting comment text that may contain label-related reasons",
    "koi_vet_date": "vetting metadata date",
    "koi_tce_delivname": "TCE delivery metadata",
    "koi_datalink_dvs": "data-product link, not a model feature",
    "koi_disp_prov": "disposition provenance metadata",
    "koi_parm_prov": "parameter provenance metadata",
    "koi_datalink_dvr": "data-product link, not a model feature",
    "koi_fpflag_nt": "false-positive vetting flag; direct leakage for false-positive labels",
    "koi_fpflag_ss": "false-positive vetting flag; direct leakage for false-positive labels",
    "koi_fpflag_co": "false-positive vetting flag; direct leakage for false-positive labels",
    "koi_fpflag_ec": "false-positive vetting flag; direct leakage for false-positive labels",
    "koi_score": "disposition confidence/score; target-leakage risk",
    "koi_fittype": "fitting-method metadata rather than a physical measurement",
}

exclusion_rows = []
for column, reason in manual_exclusion_reasons.items():
    if column in working_df.columns:
        exclusion_rows.append({"Column": column, "Reason for Exclusion": reason})

excluded_columns = {row["Column"] for row in exclusion_rows}
exclusion_table = pd.DataFrame(exclusion_rows).sort_values("Column")
display(exclusion_table)

In [ ]:
rows_before_target_drop = len(working_df)
modeling_df = working_df.dropna(subset=[target_column]).copy()
print(f"Rows removed because target is missing: {rows_before_target_drop - len(modeling_df):,}")

candidate_feature_columns = [
    column for column in modeling_df.columns
    if column not in excluded_columns
]

all_missing_columns = [
    column for column in candidate_feature_columns
    if modeling_df[column].isna().all()
]

constant_columns = [
    column for column in candidate_feature_columns
    if column not in all_missing_columns and modeling_df[column].nunique(dropna=True) <= 1
]

raw_categorical_columns = modeling_df[candidate_feature_columns].select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

high_cardinality_categorical = [
    column for column in raw_categorical_columns
    if column not in all_missing_columns + constant_columns
    and modeling_df[column].nunique(dropna=True) > 20
]

screened_out_columns = set(
    all_missing_columns + constant_columns + high_cardinality_categorical
)

feature_columns = [
    column for column in candidate_feature_columns
    if column not in screened_out_columns
]

feature_screening = pd.DataFrame([
    {"Column": column, "Reason": "all values are missing"}
    for column in all_missing_columns
] + [
    {"Column": column, "Reason": "constant or no useful variation"}
    for column in constant_columns
] + [
    {"Column": column, "Reason": "high-cardinality categorical feature; unsuitable for this baseline"}
    for column in high_cardinality_categorical
])

if not feature_screening.empty:
    display(feature_screening.sort_values("Column").reset_index(drop=True))
else:
    print("No additional columns removed during feature screening.")

In [ ]:
X = modeling_df[feature_columns].copy()
y = modeling_df[target_column].astype(str).copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Number of modelling features: {len(feature_columns)}")
print(f"Number of numerical features: {len(numeric_features)}")
print(f"Number of categorical features: {len(categorical_features)}")
print("Numerical feature names:")
print(numeric_features)
print("Categorical feature names:")
print(categorical_features)

In [ ]:
correlation_columns = [column for column in available_important_numeric if column in numeric_features]

if engineered_features:
    correlation_columns.extend([column for column in engineered_features if column in numeric_features])

correlation_columns = list(dict.fromkeys(correlation_columns))[:12]

if len(correlation_columns) >= 2:
    plt.figure(figsize=(11, 8))
    correlation_matrix = modeling_df[correlation_columns].corr(numeric_only=True)
    sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, annot=False)
    plt.title("Correlation Heatmap for Selected Numerical Features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric modelling features for a correlation heatmap.")

In [ ]:
relationship_columns = [
    column for column in ["koi_model_snr", "koi_prad", "koi_period", "koi_duration"]
    if column in modeling_df.columns
]

for column in relationship_columns[:2]:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=modeling_df, x=target_column, y=column, palette="Set3")
    plt.title(f"{column} by KOI Disposition")
    plt.xlabel("KOI disposition")
    plt.ylabel(column)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

> STUDENT INTERPRETATION TODO: Explain what each feature-target plot indicates. Avoid claiming causation; describe only what the visual output supports.

## 10. Train/Test Split

In [ ]:
stratify_values = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_values,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

split_distribution = pd.concat(
    [
        y_train.value_counts(normalize=True).mul(100).round(2).rename("train_percent"),
        y_test.value_counts(normalize=True).mul(100).round(2).rename("test_percent"),
    ],
    axis=1,
)

display(split_distribution)

## 11. Preprocessing Pipeline

In [ ]:
try:
    categorical_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", categorical_encoder),
])

preprocessing_steps = []
if numeric_features:
    preprocessing_steps.append(("numeric", numeric_transformer, numeric_features))
if categorical_features:
    preprocessing_steps.append(("categorical", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(
    transformers=preprocessing_steps,
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Preprocessing is defined but not fitted yet. It will be fitted only inside model pipelines using X_train.")

The preprocessing pipeline prevents data leakage because imputation, scaling, and encoding are fitted only inside the training pipeline. No transformer is fitted on the full dataset before the train/test split.

## 12. Gaussian Naive Bayes

Gaussian Naive Bayes uses Bayes' theorem to estimate the probability of each class given the observed features. It makes a conditional independence assumption: after the class is known, each feature is treated as independent of the others.

The Gaussian version assumes continuous numeric features can be modelled with a normal distribution within each class. This makes it a simple baseline for tabular classification, but correlated features can violate the independence assumption. We should not claim the KOI features perfectly satisfy this assumption without evidence.

In [ ]:
def calculate_ovr_roc_auc(y_true: pd.Series, probabilities: np.ndarray, classes: np.ndarray) -> float:
    """Calculate ROC-AUC with probability columns aligned to model.classes_."""
    if len(classes) == 2:
        positive_class_probabilities = probabilities[:, 1]
        return roc_auc_score(y_true, positive_class_probabilities)

    y_true_binarized = label_binarize(y_true, classes=classes)
    return roc_auc_score(
        y_true_binarized,
        probabilities,
        multi_class="ovr",
        average="weighted",
    )


def evaluate_classifier(model_name: str, estimator: Pipeline, X_eval: pd.DataFrame, y_eval: pd.Series):
    """Return a metrics row and the predictions needed for plots."""
    predictions = estimator.predict(X_eval)
    probabilities = estimator.predict_proba(X_eval)
    classes = estimator.classes_

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision_Weighted": precision_score(y_eval, predictions, average="weighted", zero_division=0),
        "Recall_Weighted": recall_score(y_eval, predictions, average="weighted", zero_division=0),
        "F1_Weighted": f1_score(y_eval, predictions, average="weighted", zero_division=0),
        "ROC_AUC_OvR": calculate_ovr_roc_auc(y_eval, probabilities, classes),
    }

    print(f"Classification report: {model_name}")
    print(classification_report(y_eval, predictions, labels=classes, zero_division=0))

    return metrics, predictions, probabilities


def plot_confusion_matrix(y_true, predictions, classes, title: str):
    matrix = confusion_matrix(y_true, predictions, labels=classes)
    display_plot = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=classes)
    display_plot.plot(cmap="Blues", xticks_rotation=25)
    plt.title(title)
    plt.xlabel("Predicted label")
    plt.ylabel("Actual label")
    plt.tight_layout()
    plt.show()

In [ ]:
gnb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GaussianNB()),
])

gnb_pipeline.fit(X_train, y_train)

gnb_metrics, gnb_predictions, gnb_probabilities = evaluate_classifier(
    "Gaussian Naive Bayes",
    gnb_pipeline,
    X_test,
    y_test,
)

display(pd.DataFrame([gnb_metrics]).round(4))

## 13. Decision Tree Classifier

In [ ]:
tree_baseline_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

tree_baseline_pipeline.fit(X_train, y_train)

tree_baseline_metrics, tree_baseline_predictions, tree_baseline_probabilities = evaluate_classifier(
    "Decision Tree Classifier - Baseline",
    tree_baseline_pipeline,
    X_test,
    y_test,
)

display(pd.DataFrame([tree_baseline_metrics]).round(4))

## 14. Hyperparameter Tuning

In [ ]:
tree_tuning_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [None, 3, 5, 8, 10],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
}

tree_grid_search = GridSearchCV(
    estimator=tree_tuning_pipeline,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
)

tree_grid_search.fit(X_train, y_train)

tuned_tree_pipeline = tree_grid_search.best_estimator_

tuned_tree_metrics, tuned_tree_predictions, tuned_tree_probabilities = evaluate_classifier(
    "Decision Tree Classifier - Tuned",
    tuned_tree_pipeline,
    X_test,
    y_test,
)

tuning_summary = pd.DataFrame([
    {"Item": "Best Parameters", "Value": tree_grid_search.best_params_},
    {"Item": "Best CV Weighted F1", "Value": round(tree_grid_search.best_score_, 4)},
    {"Item": "Baseline Test Weighted F1", "Value": round(tree_baseline_metrics["F1_Weighted"], 4)},
    {"Item": "Tuned Test Weighted F1", "Value": round(tuned_tree_metrics["F1_Weighted"], 4)},
    {"Item": "Improvement", "Value": round(tuned_tree_metrics["F1_Weighted"] - tree_baseline_metrics["F1_Weighted"], 4)},
])

display(tuning_summary)

## 15. Evaluation

In [ ]:
plot_confusion_matrix(
    y_test,
    gnb_predictions,
    gnb_pipeline.classes_,
    "Confusion Matrix - Gaussian Naive Bayes",
)

plot_confusion_matrix(
    y_test,
    tree_baseline_predictions,
    tree_baseline_pipeline.classes_,
    "Confusion Matrix - Decision Tree Baseline",
)

plot_confusion_matrix(
    y_test,
    tuned_tree_predictions,
    tuned_tree_pipeline.classes_,
    "Confusion Matrix - Decision Tree Tuned",
)

In [ ]:
# Optional multiclass One-vs-Rest ROC curves for the tuned Decision Tree.
classes = tuned_tree_pipeline.classes_
y_test_binarized = label_binarize(y_test, classes=classes)

plt.figure(figsize=(8, 6))
for class_index, class_name in enumerate(classes):
    false_positive_rate, true_positive_rate, _ = roc_curve(
        y_test_binarized[:, class_index],
        tuned_tree_probabilities[:, class_index],
    )
    plt.plot(false_positive_rate, true_positive_rate, label=f"{class_name}")

plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.title("Tuned Decision Tree - OvR ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Model Comparison

In [ ]:
comparison_table = pd.DataFrame([
    gnb_metrics,
    tree_baseline_metrics,
    tuned_tree_metrics,
]).round(4)

comparison_table = comparison_table.sort_values("F1_Weighted", ascending=False).reset_index(drop=True)
display(comparison_table)

## 17. Review 1 Conclusions / Student Interpretation

> STUDENT INTERPRETATION TODO: Compare the implemented models using the final table. Discuss accuracy, weighted F1, ROC-AUC, confusion matrices, and any visible class imbalance. Keep the explanation based on the actual notebook outputs.

### Decision Tree Visualisation

In [ ]:
fitted_preprocessor = tuned_tree_pipeline.named_steps["preprocess"]
fitted_tree = tuned_tree_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

plt.figure(figsize=(24, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=fitted_tree.classes_,
    max_depth=3,
    filled=True,
    rounded=True,
    impurity=False,
    fontsize=8,
)
plt.title("Tuned Decision Tree - Top Levels Only")
plt.tight_layout()
plt.show()

The tree visualisation is truncated with `max_depth=3` only for readability. The trained model itself still uses the best parameters selected by `GridSearchCV`.

### Feature Importance

In [ ]:
feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": fitted_tree.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=feature_importance.head(15),
    y="Feature",
    x="Importance",
    color="#2563eb",
)
plt.title("Top Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

> STUDENT INTERPRETATION TODO: Explain which measurements appear most influential and why that may make sense after checking the feature importance table.

## 18. Remaining Classification Part-A Models

### Logistic Regression

TODO - teammate implementation.

### K-Nearest Neighbors

TODO - teammate implementation.

### Support Vector Machine

TODO - teammate implementation.

All three models should use the same train/test split and should be added to the common comparison table. Do not create fake metrics before implementing them.